## HDX Signals 


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data_clean/final_data.csv')
print(f"Data loaded successfully: {df.shape}")

Data loaded successfully: (17424, 100)


In [2]:
# HDX base variables
hdx_cols = ['hdx_alert_level', 'hdx_value']

print(df[hdx_cols].describe(include='all').round(2))

print(f"\nMissing values:")
print(df[hdx_cols].isnull().sum())
print(f"\nMissing percentage:")
print((df[hdx_cols].isnull().sum() / len(df) * 100).round(2))

       hdx_alert_level   hdx_value
count            17424    17424.00
unique               5         NaN
top                 []         NaN
freq             16475         NaN
mean               NaN     2911.53
std                NaN    60973.19
min                NaN        0.00
25%                NaN        0.00
50%                NaN        0.00
75%                NaN        0.00
max                NaN  4008840.81

Missing values:
hdx_alert_level    0
hdx_value          0
dtype: int64

Missing percentage:
hdx_alert_level    0.0
hdx_value          0.0
dtype: float64


In [3]:
# Distribution of alert level
print(df['hdx_alert_level'].value_counts(dropna=False))

hdx_alert_level
[]                                    16475
['Medium concern']                      587
['High concern']                        294
['Medium concern', 'High concern']       55
['High concern', 'Medium concern']       13
Name: count, dtype: int64


In [4]:
# Descriptive statistics for hdx_value
print(df['hdx_value'].describe().round(2))
print(f"\nMissing values: {df['hdx_value'].isnull().sum()}")
print(f"Missing percentage: {(df['hdx_value'].isnull().sum() / len(df) * 100).round(2)}%")

count      17424.00
mean        2911.53
std        60973.19
min            0.00
25%            0.00
50%            0.00
75%            0.00
max      4008840.81
Name: hdx_value, dtype: float64

Missing values: 0
Missing percentage: 0.0%


In [5]:
# Find the observation with the maximum hdx_value
print(df[df['hdx_value'] == df['hdx_value'].max()][['iso3', 'month', 'hdx_alert_level', 'hdx_value']])

     iso3       month                     hdx_alert_level     hdx_value
3012  CHN  2021-07-01  ['Medium concern', 'High concern']  4.008841e+06


This is possible an oultlier.

In [7]:
# Filter only Medium and High concern
df['hdx_alert_simple'] = df['hdx_alert_level'].apply(
    lambda x: 'High concern' if 'High concern' in str(x) else 
              ('Medium concern' if 'Medium concern' in str(x) else 'No alert')
)

# Crosstab only for Medium and High concern
mask = df['hdx_alert_simple'] != 'No alert'
pd.crosstab(df[mask]['allocation-eligible'], df[mask]['hdx_alert_simple'], 
            normalize='index').round(3) * 100

hdx_alert_simple,High concern,Medium concern
allocation-eligible,,
0,34.0,66.0
1,53.2,46.8


In [8]:
# De todos los allocation-eligible, cuantos tienen alguna alerta
pd.crosstab(df['allocation-eligible'], df['hdx_alert_simple'], normalize='index').round(3) * 100

hdx_alert_simple,High concern,Medium concern,No alert
allocation-eligible,,,
0,1.5,2.9,95.5
1,15.9,14.0,70.1


The crosstab reveals two important findings about HDX Signals coverage and predictive value. First, the variable has very sparse coverage , 95.5% of non-eligible country-months and 70.1% of eligible country-months have no alert recorded, reflecting that HDX only flags a small subset of humanitarian situations.

Second, despite this limitation, there is a clear difference between groups. Allocation-eligible country-months are almost 7 times more likely to have an alert than non-eligible ones (29.9% vs 4.4%), and show a higher proportion of "High concern" alerts (15.9% vs 1.5%). This suggests that when HDX does register an alert, it tends to coincide with genuine crisis situations. Given its sparse coverage, HDX Signals should be treated as a complementary signal rather than a primary predictor in the early warning model.